In [ ]:
import kagglehub

data = kagglehub.competition_download(
    'inter-uni-datathon-stream-2-beijing-multi-site-air-quality',
    output_dir='data'
)

print("Path to competition files:", data)

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("Datathon26")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")

    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "2g")

    .getOrCreate()
)

df = spark.read.csv("data/train.csv", header=True, inferSchema=True)
test_df=spark.read.csv("data/test(1).csv", header=True, inferSchema=True)

In [ ]:
import math
from itertools import product

import numpy as np
import pandas as pd
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
import lightgbm as lgb

# ---------------------------------------------------------------------------
# CONFIGURATION & GLOBAL SETTINGS
# ---------------------------------------------------------------------------
# Set to True for a quick single-fold run to test changes.
FAST = False 

# If True, predictors at time t+1 are used.
USE_LEADS = True  

TARGET = "PM2_5_next_hour"
TS_COL = "observation_timestamp"
STATION = "station"
ID_COL = "id"

# Bounds for the target variable based on domain knowledge
Y_MIN, Y_MAX = 2.0, 999.0
N_TEST_EXPECTED = 51063

TUNE_RECENCY = True
# Half-life grid (in days) determines how quickly older data loses importance
HALFLIFE_GRID = [None, 1095, 730, 365, 180] 
CALIBRATE = True
SEEDS = [42, 202]
NUM_THREADS = 8 

# Compass directions mapped to degrees for trigonometric wind features
WD_DEG = {
    "N": 0.0, "NNE": 22.5, "NE": 45.0, "ENE": 67.5, "E": 90.0, "ESE": 112.5,
    "SE": 135.0, "SSE": 157.5, "S": 180.0, "SSW": 202.5, "SW": 225.0,
    "WSW": 247.5, "W": 270.0, "WNW": 292.5, "NW": 315.0, "NNW": 337.5
}
BASE_NUM = ["PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
POLL = ["PM10", "CO", "NO2", "SO2", "O3"]

# Optimize PySpark performance
try:
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    spark.conf.set("spark.sql.shuffle.partitions", "16")
except Exception:
    pass


# ===========================================================================
# 1. FEATURE ENGINEERING
# ===========================================================================
def add_features(sdf: DataFrame) -> DataFrame:

    # Generates time-series features using PySpark window functions.

    # 1. Base Time Variables
    _ts = F.to_timestamp(F.col(TS_COL))
    sdf = sdf.withColumn("ts", _ts) \
             .withColumn("hidx", (F.unix_timestamp(_ts) / F.lit(3600)).cast("long"))
    
    # Ensure numerical consistency between train and test
    for col_name in BASE_NUM: 
        sdf = sdf.withColumn(col_name, F.col(col_name).cast("double"))

    # 2. Cyclical Time Features (Hour, Day, Month)
    tp = 2.0 * math.pi
    sdf = sdf.withColumn("f_hour", F.hour("ts")) \
             .withColumn("f_doy", F.dayofyear("ts")) \
             .withColumn("f_dow", F.dayofweek("ts")) \
             .withColumn("f_month", F.month("ts"))
    
    sdf = sdf.withColumn("hour_sin", F.sin(F.col("f_hour") * F.lit(tp / 24))) \
             .withColumn("hour_cos", F.cos(F.col("f_hour") * F.lit(tp / 24))) \
             .withColumn("doy_sin", F.sin(F.col("f_doy") * F.lit(tp / 365.25))) \
             .withColumn("doy_cos", F.cos(F.col("f_doy") * F.lit(tp / 365.25))) \
             .withColumn("is_heating", ((F.col("f_month") >= 11) | (F.col("f_month") <= 3)).cast("int"))

    # 3. Meteorological Features (Wind Vectors & Humidity)
    wd_map = F.create_map([F.lit(x) for kv in WD_DEG.items() for x in kv])
    sdf = sdf.withColumn("wd_deg", F.element_at(wd_map, F.upper(F.trim(F.col("wd"))))) \
             .withColumn("wd_rad", F.radians("wd_deg"))
    
    # Calculate Relative Humidity (RH) using the Magnus-Tetens approximation
    a, b = 17.625, 243.04
    sdf = sdf.withColumn("wd_sin", F.sin("wd_rad")) \
             .withColumn("wd_cos", F.cos("wd_rad")) \
             .withColumn("wind_u", -F.col("WSPM") * F.sin("wd_rad")) \
             .withColumn("wind_v", -F.col("WSPM") * F.cos("wd_rad")) \
             .withColumn("dew_dep", F.col("TEMP") - F.col("DEWP")) \
             .withColumn("RH", F.lit(100.0) * F.exp(
                 (F.lit(a) * F.col("DEWP")) / (F.lit(b) + F.col("DEWP")) -
                 (F.lit(a) * F.col("TEMP")) / (F.lit(b) + F.col("TEMP"))))

    # 4. Backward-Looking Window Functions (Lags & Rolling Aggregates)
    w_station = Window.partitionBy(STATION).orderBy("hidx")
    
    # Helper functions for clean window expression generation
    def lag(c, k): return F.max(F.col(c)).over(w_station.rangeBetween(-k, -k)).alias(f"{c}_lag{k}")
    def roll_mean(c, k): return F.avg(F.col(c)).over(w_station.rangeBetween(-(k - 1), 0)).alias(f"{c}_rm{k}")
    def roll_max(c, k): return F.max(F.col(c)).over(w_station.rangeBetween(-(k - 1), 0)).alias(f"{c}_rmax{k}")
    def roll_min(c, k): return F.min(F.col(c)).over(w_station.rangeBetween(-(k - 1), 0)).alias(f"{c}_rmin{k}")
    def roll_sum(c, k): return F.sum(F.col(c)).over(w_station.rangeBetween(-(k - 1), 0)).alias(f"{c}_rs{k}")
    def roll_std(c, k): return F.stddev(F.col(c)).over(w_station.rangeBetween(-(k - 1), 0)).alias(f"{c}_rsd{k}")

    lag_features = [lag("PM10", k) for k in (1, 2, 3, 4, 6, 8, 12, 18, 24, 36, 48)]
    for c in ("CO", "NO2"):
        lag_features += [lag(c, k) for k in (1, 2, 3, 6, 12, 24)]
    for c in ("SO2", "O3", "TEMP", "PRES", "DEWP", "WSPM", "RH"):
        lag_features += [lag(c, k) for k in (1, 3, 6, 24)]
        
    for c in ("PM10", "CO", "NO2"):
        lag_features += [roll_mean(c, k) for k in (3, 6, 12, 24, 48)] + [roll_std(c, 24)]
        
    lag_features += [roll_max("PM10", 24), roll_min("PM10", 24), roll_max("CO", 24)]
    lag_features += [roll_mean("WSPM", 3), roll_mean("WSPM", 6), roll_mean("WSPM", 24), 
                     roll_min("WSPM", 12), roll_min("WSPM", 24)]
    lag_features += [roll_mean("RH", 6), roll_mean("RH", 24), roll_mean("wind_u", 6), 
                     roll_mean("wind_v", 6), roll_mean("wind_u", 24), roll_mean("wind_v", 24)]
    lag_features += [roll_sum("RAIN", 3), roll_sum("RAIN", 6), roll_sum("RAIN", 24)]
    
    sdf = sdf.select("*", *lag_features)

    # 5. Lead Features
    if USE_LEADS:
        def lead(c, k): return F.max(F.col(c)).over(w_station.rangeBetween(k, k)).alias(f"{c}_lead{k}")
        
        lead_features = [lead(c, 1) for c in ("PM10", "CO", "NO2", "SO2", "O3", 
                                              "TEMP", "PRES", "DEWP", "WSPM", 
                                              "RAIN", "RH", "wind_u", "wind_v")]
        for c in ("PM10", "CO", "NO2"):
            lead_features += [lead(c, 2), lead(c, 3)]
            
        lead_features += [
            F.avg(F.col("PM10")).over(w_station.rangeBetween(0, 2)).alias("PM10_fwd3"),
            F.avg(F.col("CO")).over(w_station.rangeBetween(0, 2)).alias("CO_fwd3"),
            F.max(F.col("PM10")).over(w_station.rangeBetween(0, 3)).alias("PM10_fwdmax4")
        ]
        sdf = sdf.select("*", *lead_features)

    # 6. Trend, Acceleration, and City-Wide States
    w_city = Window.partitionBy("hidx")
    trend_features = [(F.col("PM10") - F.col(f"PM10_lag{k}")).alias(f"PM10_d{k}") for k in (1, 2, 3, 6, 12, 24)]
    trend_features += [((F.col("PM10") - F.col("PM10_lag1")) - 
                        (F.col("PM10_lag1") - F.col("PM10_lag2"))).alias("PM10_acc")]
    
    for c in ("CO", "NO2"):
        trend_features += [(F.col(c) - F.col(f"{c}_lag{k}")).alias(f"{c}_d{k}") for k in (1, 3, 24)]
    for c in ("SO2", "O3", "TEMP", "PRES", "DEWP", "WSPM"):
        trend_features += [(F.col(c) - F.col(f"{c}_lag{k}")).alias(f"{c}_d{k}") for k in (3, 24)]
        
    for c in POLL:
        trend_features += [F.avg(c).over(w_city).alias(f"city_{c}"), F.stddev(c).over(w_city).alias(f"citysd_{c}")]
        
    trend_features += [F.max("PM10").over(w_city).alias("city_max_PM10"), 
                       F.count(F.lit(1)).over(w_city).alias("city_n")]
    
    sdf = sdf.select("*", *trend_features)

    # 7. Ratios, Proxies, and Physics-Based Interactions
    interaction_feats = []
    for c in POLL:
        interaction_feats += [(F.col(c) - F.col(f"city_{c}")).alias(f"{c}_dev"),
                              (F.col(c) / (F.col(f"city_{c}") + F.lit(1e-3))).alias(f"{c}_rat")]
                              
    for c in ("city_PM10", "city_CO"):
        interaction_feats += [F.max(F.col(c)).over(w_station.rangeBetween(-1, -1)).alias(f"{c}_lag1"),
                              F.max(F.col(c)).over(w_station.rangeBetween(-3, -3)).alias(f"{c}_lag3"),
                              F.avg(F.col(c)).over(w_station.rangeBetween(-23, 0)).alias(f"{c}_rm24")]
                              
    interaction_feats += [F.log1p(F.greatest(F.col(c), F.lit(0.0))).alias(f"log_{c}") for c in POLL]
    
    # PM2.5 Proxy: PM2.5 is highly correlated with PM10 * hygroscopic ratio (f(RH)).
    # We provide several functional forms of this interaction to let the trees decide.
    interaction_feats += [
        (F.col("CO") / (F.col("PM10") + F.lit(1.0))).alias("CO_over_PM10"),
        (F.col("NO2") / (F.col("O3") + F.lit(1.0))).alias("NO2_over_O3"),
        (F.col("SO2") / (F.col("PM10") + F.lit(1.0))).alias("SO2_over_PM10"),
        (F.col("PM10") / (F.col("WSPM") + F.lit(0.1))).alias("PM10_vent"),
        (F.col("PM10") * F.col("RH") / F.lit(100.0)).alias("PM10_x_RH"),
        (F.col("PM10") * F.pow(F.col("RH") / F.lit(100.0), F.lit(2.0))).alias("PM10_x_RH2"),
        (F.col("PM10") / F.greatest(F.lit(0.02), F.lit(1.02) - F.col("RH") / F.lit(100.0))).alias("PM10_hygro"),
        (F.col("PM10") * F.col("RH_rm24") / F.lit(100.0)).alias("PM10_x_RH24"),
        (F.col("PM10_lag1") * F.col("RH_lag1") / F.lit(100.0)).alias("PM10lag1_x_RH"),
        (F.col("city_PM10") * F.col("RH") / F.lit(100.0)).alias("cityPM10_x_RH"),
        (F.col("CO") * F.col("RH") / F.lit(100.0)).alias("CO_x_RH"),
        (F.col("PM10") * F.col("dew_dep")).alias("PM10_x_dewdep"),
        sum(F.col(c).isNull().cast("int") for c in BASE_NUM).alias("n_missing")
    ]
    
    if USE_LEADS:
        for c in ("city_PM10", "city_CO", "city_NO2"):
            interaction_feats.append(F.max(F.col(c)).over(w_station.rangeBetween(1, 1)).alias(f"{c}_lead1"))
            
        # Target hour proxy evaluation 
        interaction_feats += [
            (F.col("PM10_lead1") * F.col("RH_lead1") / F.lit(100.0)).alias("PM10lead_x_RHlead"),
            (F.col("PM10_lead1") / F.greatest(F.lit(0.02), F.lit(1.02) - F.col("RH_lead1") / F.lit(100.0))).alias("PM10lead_hygro"),
            F.log1p(F.greatest(F.col("PM10_lead1"), F.lit(0.0))).alias("log_PM10_lead1"),
            F.log1p(F.greatest(F.col("CO_lead1"), F.lit(0.0))).alias("log_CO_lead1"),
            (F.col("PM10_lead1") - F.col("PM10")).alias("PM10_dlead1"),
            (F.col("CO_lead1") - F.col("CO")).alias("CO_dlead1"),
            (F.col("NO2_lead1") - F.col("NO2")).alias("NO2_dlead1"),
            (F.col("CO_lead1") / (F.col("PM10_lead1") + F.lit(1.0))).alias("CO_over_PM10_lead"),
            (F.col("PM10_lead1") / (F.col("WSPM_lead1") + F.lit(0.1))).alias("PM10lead_vent")
        ]

    sdf = sdf.select("*", *interaction_feats)

    # 8. Final Delta Features
    delta_feats = [
        (F.col("city_PM10") - F.col("city_PM10_lag1")).alias("city_PM10_d1"),
        (F.col("city_PM10") - F.col("city_PM10_lag3")).alias("city_PM10_d3"),
        (F.col("city_CO") - F.col("city_CO_lag1")).alias("city_CO_d1")
    ]
    if USE_LEADS:
        delta_feats += [
            (F.col("PM10_lead1") - F.col("city_PM10_lead1")).alias("PM10lead_dev"),
            (F.col("PM10_lead1") / (F.col("city_PM10_lead1") + F.lit(1e-3))).alias("PM10lead_rat"),
            (F.col("city_PM10_lead1") - F.col("city_PM10")).alias("city_PM10_dlead1"),
            (F.col("city_CO_lead1") - F.col("city_CO")).alias("city_CO_dlead1")
        ]
        
    return sdf.select("*", *delta_feats)


# ===========================================================================
# 2. MATRIX PREPARATION
# ===========================================================================

# Union train and test BEFORE feature engineering so test rows can access 
# the historical trailing data required for lag calculation.
test_marked = test_df.withColumn(TARGET, F.lit(None).cast("double")).withColumn("_is_test", F.lit(1))
full_df = df.withColumn("_is_test", F.lit(0)).unionByName(test_marked)

print("Engineering features (this may take a moment)...")
pdf = add_features(full_df).toPandas()

# Clean up columns and memory types
DROP_COLS = {ID_COL, TS_COL, TARGET, "ts", "wd", "wd_rad", "_is_test", "hidx", "year", "day"}
pdf[STATION] = pdf[STATION].astype("category")
FEATS = [c for c in pdf.columns if c not in DROP_COLS]

for c in FEATS:
    if c != STATION:
        if pdf[c].dtype == object:
            pdf[c] = pd.to_numeric(pdf[c], errors="coerce")
        if pdf[c].dtype == np.float64:
            pdf[c] = pdf[c].astype(np.float32)

# Separate back into train and test pandas DataFrames
train_pdf = pdf[(pdf["_is_test"] == 0) & pdf[TARGET].notna()].sort_values("hidx").reset_index(drop=True)
test_pdf = pdf[pdf["_is_test"] == 1].reset_index(drop=True)

X_train, y_train, ts_train = train_pdf[FEATS], train_pdf[TARGET].astype(np.float64).values, train_pdf["ts"]
age_days = (train_pdf["hidx"].max() - train_pdf["hidx"].values) / 24.0
lagcols = [c for c in FEATS if "_lag" in c or "_rm" in c]

print(f"Train Rows: {len(train_pdf):,} | Test Rows: {len(test_pdf):,} | Features: {len(FEATS)}")
print(f"Lag Coverage — Train: {train_pdf[lagcols].notna().mean().mean():.3f}, "
      f"Test: {test_pdf[lagcols].notna().mean().mean():.3f}")


# ===========================================================================
# 3. MODELING & CROSS-VALIDATION
# ===========================================================================
COMMON_PARAMS = dict(
    objective="regression", metric="rmse", bagging_fraction=0.8, bagging_freq=1,
    max_bin=255, num_threads=NUM_THREADS, force_col_wise=True, verbosity=-1
)
LR = 0.15 if FAST else 0.03

# Model specifications: Name, Parameters, Use_Log_Target
# We use two capacities (high/low leaves) and one model that predicts log(target)
SPECS = [
    ("lo_L2", dict(COMMON_PARAMS, learning_rate=LR, num_leaves=63, min_data_in_leaf=120, feature_fraction=0.5, lambda_l2=20.0), False),
    ("hi_L2", dict(COMMON_PARAMS, learning_rate=LR, num_leaves=160, min_data_in_leaf=40, feature_fraction=0.65, lambda_l2=6.0), False),
    ("lo_log", dict(COMMON_PARAMS, learning_rate=LR, num_leaves=63, min_data_in_leaf=120, feature_fraction=0.5, lambda_l2=20.0), True)
]
if FAST:
    SPECS = SPECS[:1]

def calculate_rmse(actuals, preds): 
    return float(np.sqrt(np.mean((np.asarray(actuals) - np.asarray(preds)) ** 2)))

def get_season_folds(timestamps):

    # Creates temporal CV folds. Trains through Aug 31st, validates Sept 1st -> Feb 28th.

    out = []
    for yr in sorted(timestamps.dt.year.unique()):
        cut, end = pd.Timestamp(f"{yr}-09-01"), pd.Timestamp(f"{yr + 1}-03-01")
        train_idx = np.where(timestamps < cut)[0]
        val_idx = np.where((timestamps >= cut) & (timestamps < end))[0]
        if len(train_idx) > 30000 and len(val_idx) > 5000:
            out.append((train_idx, val_idx))
    return out

FOLDS = get_season_folds(ts_train)[-1:] if FAST else get_season_folds(ts_train)
print(f"\nCreated {len(FOLDS)} folds: " + " | ".join(f"Train {len(t):,} / Val {len(v):,}" for t, v in FOLDS))

def get_weights(half_life): 

    # Decays sample weights exponentially based on age to prioritize recent trends.

    if half_life is None:
        return np.ones(len(train_pdf))
    return 0.5 ** (age_days / half_life)

def run_lgb(train_idx, val_idx, weights, is_log_target, params, rounds=8000, early_stop=150):

    # Wrapper to train a single LightGBM model and return predictions.

    y_tr = np.log1p(y_train[train_idx]) if is_log_target else y_train[train_idx]
    y_va = np.log1p(y_train[val_idx]) if is_log_target else y_train[val_idx]
    
    d_train = lgb.Dataset(X_train.iloc[train_idx], y_tr, weight=weights[train_idx], categorical_feature=[STATION])
    d_val = lgb.Dataset(X_train.iloc[val_idx], y_va, reference=d_train)
    
    model = lgb.train(
        params, 
        d_train, 
        num_boost_round=rounds,
        valid_sets=[d_val],
        callbacks=[lgb.early_stopping(early_stop, verbose=False)]
    )
    
    raw_preds = model.predict(X_train.iloc[val_idx], num_iteration=model.best_iteration)
    final_preds = np.expm1(raw_preds) if is_log_target else raw_preds
    return model, np.clip(final_preds, Y_MIN, Y_MAX)

# ---- Recency Half-Life Tuning ---------------------------------------------
HALFLIFE = None
if TUNE_RECENCY and not FAST:
    print("\nTuning recency half-life...")
    train_idx, val_idx = FOLDS[-1]
    fast_params = dict(SPECS[0][1], learning_rate=0.09)
    best = (np.inf, None)
    
    for hl in HALFLIFE_GRID:
        _, preds = run_lgb(train_idx, val_idx, get_weights(hl), False, fast_params, rounds=2000, early_stop=80)
        score = calculate_rmse(y_train[val_idx], preds)
        print(f"  {str(hl):>5} days -> RMSE: {score:.4f}")
        best = min(best, (score, hl))
        
    HALFLIFE = best[1]
    print(f"  Chosen half-life: {HALFLIFE}")
    
FINAL_WEIGHTS = get_weights(HALFLIFE)

# ---- Cross Validation Loop ------------------------------------------------
oof_preds = np.full((len(train_pdf), len(SPECS)), np.nan)
rounds_for_spec = {}

for spec_idx, (name, params, is_log_target) in enumerate(SPECS):
    print(f"\n== Training Spec: {name} ==")
    best_iters, fold_sizes = [], []
    
    for fold_idx, (train_idx, val_idx) in enumerate(FOLDS):
        model, preds = run_lgb(train_idx, val_idx, FINAL_WEIGHTS, is_log_target, params)
        oof_preds[val_idx, spec_idx] = preds
        
        best_iters.append(model.best_iteration)
        fold_sizes.append(len(train_idx))
        
        val_rmse = calculate_rmse(y_train[val_idx], preds)
        val_month = ts_train.iloc[val_idx[0]]
        print(f"  Fold {fold_idx} ({val_month:%Y-%m}): RMSE {val_rmse:.4f}  | Iters {model.best_iteration}")
        
    # Weight iteration counts by fold size to ensure the final refit scale is accurate
    rounds_for_spec[name] = int(np.average(best_iters, weights=fold_sizes))
    last_val_idx = FOLDS[-1][1]
    print(f"  Last Fold RMSE: {calculate_rmse(y_train[last_val_idx], oof_preds[last_val_idx, spec_idx]):.4f} | Avg Rounds: {rounds_for_spec[name]}")

# Filter out rows that were never in a validation set
mask = ~np.isnan(oof_preds).any(axis=1)
Valid_Preds, Valid_Actuals = oof_preds[mask], y_train[mask]

# ---- Ensembling: Blend weights over a simplex grid ------------------------
if len(SPECS) > 1:
    grid = np.arange(0, 1.0001, 0.05)
    best_weights, best_error = None, np.inf
    
    for weight_combo in product(grid, repeat=len(SPECS) - 1):
        if sum(weight_combo) > 1.0001:
            continue
            
        full_weights = np.array(list(weight_combo) + [1 - sum(weight_combo)])
        error = calculate_rmse(Valid_Actuals, Valid_Preds @ full_weights)
        if error < best_error:
            best_weights, best_error = full_weights, error
            
    weight_str = "  ".join(f"{s[0]}={v:.2f}" for s, v in zip(SPECS, best_weights))
    print(f"\nOptimized Blend Weights: {weight_str} -> RMSE: {best_error:.4f}")
else:
    best_weights, best_error = np.array([1.0]), calculate_rmse(Valid_Actuals, Valid_Preds[:, 0])

blended_oof = Valid_Preds @ best_weights

# ---- Recalibration --------------------------------------------------------
# Fixes consistent under/over prediction tendencies, but only applied if it
# generalizes in a leave-one-fold-out check to prevent overfitting.
Calib_B0, Calib_B1 = 0.0, 1.0

if CALIBRATE and len(FOLDS) >= 2:
    print("\nChecking Linear Recalibration...")
    pos = {original_idx: new_idx for new_idx, original_idx in enumerate(np.where(mask)[0])}
    gains = []
    
    for fold_idx, (_, val_idx) in enumerate(FOLDS):
        # Fit calibration on out-of-fold predictions excluding current fold
        other_idx = np.array([pos[x] for f in range(len(FOLDS)) if f != fold_idx for x in FOLDS[f][1] if x in pos])
        curr_idx = np.array([pos[x] for x in val_idx if x in pos])
        
        b1, b0 = np.polyfit(blended_oof[other_idx], Valid_Actuals[other_idx], 1)
        
        before_rmse = calculate_rmse(Valid_Actuals[curr_idx], blended_oof[curr_idx])
        after_rmse = calculate_rmse(Valid_Actuals[curr_idx], np.clip(b0 + b1 * blended_oof[curr_idx], Y_MIN, Y_MAX))
        gains.append(before_rmse - after_rmse)
        
        print(f"  Calib Fold {fold_idx}: b0 {b0:+.2f} b1 {b1:.3f} | RMSE change: {before_rmse:.4f} -> {after_rmse:.4f}")
        
    if np.mean(gains) > 0.05 and gains[-1] > 0:
        Calib_B1, Calib_B0 = np.polyfit(blended_oof, Valid_Actuals, 1)
        print(f"  Applied! Pred -> {Calib_B0:+.3f} + {Calib_B1:.4f} * Pred (Mean gain {np.mean(gains):.3f})")
    else:
        print(f"  Discarded. Didn't hold out of fold (Mean gain {np.mean(gains):.3f})")

calibrated_oof = np.clip(Calib_B0 + Calib_B1 * blended_oof, Y_MIN, Y_MAX)
print(f"\nFinal OOF Blend RMSE: {best_error:.4f} | Calibrated RMSE: {calculate_rmse(Valid_Actuals, calibrated_oof):.4f}")
print(f"Residual Bias (Pred - Actual): {blended_oof.mean() - Valid_Actuals.mean():+.2f}")


# ===========================================================================
# 4. FINAL REFIT, PREDICTION, & SUBMISSION
# ===========================================================================
# Scale boost rounds up since we are training on 100% of data instead of CV cuts
scale_factor = len(train_pdf) / len(FOLDS[-1][0])
print(f"\nRefitting on all {len(train_pdf):,} rows (Multiplier: {scale_factor:.2f})...")

test_predictions, last_model_ref = [], None

for spec_name, params, is_log_target in SPECS:
    spec_preds = []
    
    for seed in SEEDS:
        seed_params = dict(params, seed=seed, bagging_seed=seed, feature_fraction_seed=seed)
        final_dtrain = lgb.Dataset(X_train, np.log1p(y_train) if is_log_target else y_train, 
                                   weight=FINAL_WEIGHTS, categorical_feature=[STATION])
                                   
        final_rounds = max(50, int(rounds_for_spec[spec_name] * scale_factor))
        model = lgb.train(seed_params, final_dtrain, num_boost_round=final_rounds)
        
        spec_preds.append(model.predict(test_pdf[FEATS]))
        if not is_log_target:
            last_model_ref = model
            
    # Average across seeds for this spec
    avg_spec_pred = np.mean(spec_preds, axis=0)
    final_spec_pred = np.expm1(avg_spec_pred) if is_log_target else avg_spec_pred
    test_predictions.append(np.clip(final_spec_pred, Y_MIN, Y_MAX))

# Blend specifications and apply calibration
final_pred_matrix = np.column_stack(test_predictions) @ best_weights
calibrated_test_preds = np.clip(Calib_B0 + Calib_B1 * final_pred_matrix, Y_MIN, Y_MAX)

# Construct and save submission dataframe
submission_df = pd.DataFrame({ID_COL: test_pdf[ID_COL].values, TARGET: calibrated_test_preds})

# Final Sanity Checks
assert submission_df[ID_COL].nunique() == len(submission_df) == N_TEST_EXPECTED, f"Expected {N_TEST_EXPECTED} rows, got {len(submission_df)}!"
assert submission_df[TARGET].notna().all() and submission_df[TARGET].std() > 1, "Predictions are degenerate (constant or NaN)!"

submission_df.to_csv("submissionFinal.csv", index=False, float_format="%.6f")

print(f"\nSuccess! Wrote submissionFinal.csv {submission_df.shape}")
print(f"Prediction Stats -> Mean: {calibrated_test_preds.mean():.2f} | Std: {calibrated_test_preds.std():.2f} | "
      f"Min: {calibrated_test_preds.min():.1f} | Max: {calibrated_test_preds.max():.1f}")
print("(For context: Train Mean is ~78.04, Train Sept-Feb Mean is ~87.8)\n")

print("Top 25 Features by Gain:")
importance_series = pd.Series(last_model_ref.feature_importance("gain"), index=FEATS)
print(importance_series.sort_values(ascending=False).head(25).to_string())